# Atom and Molecule Basis Classes

In this notebook, we define reusable basis classes for molecular geometry (`Atom`, `Molecule`). 

The 'Atom' class is implemented as a dataclass with two core attributes: symbol and coord. The __post_init__ function converts the coordinate input into a NumPy array of floats and validates to ensure it is three-dimensional.

 The class provides a readable string representation via __str__, an atomic_number property obtained from an internal periodic-table dictionary, and an n_electrons property that currently returns just the atomic number (i. e. a neutral atom). In addition, get_distance computes the Euclidean distance between two Atom instances and raises an error if the argument is not an Atom.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Sequence, List
import numpy as np

ATOMIC_NUMBERS = {
    "H": 1, "He": 2, "Li": 3, "Be": 4, "B": 5, "C": 6, "N": 7, "O": 8, "F": 9, "Ne": 10,
    "Na": 11, "Mg": 12, "Al": 13, "Si": 14, "P": 15, "S": 16, "Cl": 17, "Ar": 18, "K": 19, "Ca": 20,
    "Sc": 21, "Ti": 22, "V": 23, "Cr": 24, "Mn": 25, "Fe": 26, "Co": 27, "Ni": 28, "Cu": 29, "Zn": 30,
    "Ga": 31, "Ge": 32, "As": 33, "Se": 34, "Br": 35, "Kr": 36, "Rb": 37, "Sr": 38, "Y": 39, "Zr": 40,
    "Nb": 41, "Mo": 42, "Tc": 43, "Ru": 44, "Rh": 45, "Pd": 46, "Ag": 47, "Cd": 48, "In": 49, "Sn": 50,
    "Sb": 51, "Te": 52, "I": 53, "Xe": 54, "Cs": 55, "Ba": 56, "La": 57, "Ce": 58, "Pr": 59, "Nd": 60,
    "Pm": 61, "Sm": 62, "Eu": 63, "Gd": 64, "Tb": 65, "Dy": 66, "Ho": 67, "Er": 68, "Tm": 69, "Yb": 70,
    "Lu": 71, "Hf": 72, "Ta": 73, "W": 74, "Re": 75, "Os": 76, "Ir": 77, "Pt": 78, "Au": 79, "Hg": 80,
    "Tl": 81, "Pb": 82, "Bi": 83, "Po": 84, "At": 85, "Rn": 86, "Fr": 87, "Ra": 88, "Ac": 89, "Th": 90,
    "Pa": 91, "U": 92, "Np": 93, "Pu": 94, "Am": 95, "Cm": 96, "Bk": 97, "Cf": 98, "Es": 99, "Fm": 100,
    "Md": 101, "No": 102, "Lr": 103, "Rf": 104, "Db": 105, "Sg": 106, "Bh": 107, "Hs": 108, "Mt": 109,
    "Ds": 110, "Rg": 111, "Cn": 112, "Nh": 113, "Fl": 114, "Mc": 115, "Lv": 116, "Ts": 117, "Og": 118,
}
ELEMENT_SYMBOLS = {atomic_number: symbol for symbol, atomic_number in ATOMIC_NUMBERS.items()}

@dataclass
class Atom:
    symbol: str
    coord: Sequence[float]

    def __post_init__(self) -> None:
        self.symbol = self.symbol.strip().capitalize()
        self.coord = np.asarray(self.coord, dtype=float)
        if self.coord.shape != (3,):
            raise ValueError("Coordinates must be a 3D vector.")

    def __str__(self) -> str:
        return f"{self.symbol} at {self.coord}"

    @classmethod
    def from_string(cls, s: str) -> "Atom":
        parts = s.split()
        if len(parts) != 4:
            raise ValueError(f"Expected 'SYMBOL x y z', got: {s!r}")

        symbol = parts[0]
        try:
            coord = np.asarray(parts[1:4], dtype=float)
        except ValueError as exc:
            raise ValueError(f"Invalid coordinates in atom string: {s!r}") from exc
        return cls(symbol, coord)


    @property
    def atomic_number(self) -> int:
        try:
            return ATOMIC_NUMBERS[self.symbol]
        except KeyError as exc:
            raise ValueError(f"Unknown element symbol: {self.symbol!r}") from exc

    @property
    def n_electrons(self) -> int:
        return self.atomic_number

    def get_distance(self, other: Atom) -> float:
        if not isinstance(other, Atom):
            raise ValueError("Distance can only be calculated between two Atom instances.")
        return float(np.linalg.norm(self.coord - other.coord))


## Some remarks on the implementation of `Atom` 

The `Atom` is decorated with `@dataclass`, which automatically generates the `__init__` method and other utility methods based on the defined attributes.  

The `__post_init__` method converts the input coordinates into a NumPy array. This ensures that the class can accept various input formats (lists, tuples, arrays), while maintaining a unique and consistent internal representation.  

The `atomic_number` property retrieves the atomic number from a predefined dictionary based on the element symbol, and the `n_electrons` property currently assumes a neutral atom by returning the atomic number.

The `get_distance` method calculates the distance between two `Atom` instances using NumPy's linear algebra functions, and it includes a type check to ensure that the method is only called with another `Atom` instance.

### Why use type hints in `Atom`?

Type hints make the class interface explicit and easier to understand: for example, `symbol: str` and `coord: Sequence[float]` immediately tell us what inputs are expected.  
They also improve IDE support (autocompletion, documentation) and allow static tools (`mypy`, `pyright`) to catch type mismatches before runtime, which is very useful in larger scientific codebases.

### How type hints are used here

- **Attributes**: `symbol: str`, `coord: Sequence[float]`
- **Method return types**: `-> None`, `-> int`, `-> float`, `-> "Atom"`
- **Parser methods** (`from_string`) clearly state they return an `Atom`.
- `Sequence[float]` is intentionally broad: it accepts lists, tuples, or arrays as input.
- In `__post_init__`, the input is converted to `np.ndarray` and validated (`shape == (3,)`), because type hints are documentation/static checks, not runtime enforcement.



## Usage of Atom class  
atom1 = Atom("C", [0.0, 0.0, 0.0])
atom2 = Atom.from_string("O 1.0 0.0 0.0")
print(atom1)  # Output: C at [0. 0. 0.]
print(atom2)  # Output: O at [1. 0. 0.]
print(atom1.atomic_number)  # Output: 6
print(atom2.n_electrons)  # Output: 8
print(atom1.get_distance(atom2))  # Output: 1.0

## Molecule class implementation 

After defining the `Atom` class, we implement a `Molecule` class that contains a list of `Atom` instances. The `Molecule` class includes methods to calculate the total number of electrons and to convert between XYZ string representations and `Molecule` instances. The `from_string` class method parses an XYZ format string to create a `Molecule` instance, while the `to_string` method generates an XYZ format string from the `Molecule` instance.

In [ ]:
from __future__ import annotations
from typing import Self

@dataclass
class Molecule:
    atoms: List[Atom]
    
    def __str__(self):
        return "\n".join(str(atom) for atom in self.atoms)
    
    @property
    def n_electrons(self):
        return sum(atom.n_electrons for atom in self.atoms)
    
    @classmethod
    def from_string(cls, xyz_string: str) -> Self:
        lines = xyz_string.strip().splitlines()
        atoms = []
        for line in lines[2:]:  # Skip the first two lines (atom count and comment)
            atom = Atom.from_string(line)
            atoms.append(atom)
        return cls(atoms)
    
    def to_string(self) -> str:
        lines = [str(len(self.atoms)), "Generated by Molecule class"]
        for atom in self.atoms:
            line = f"{atom.symbol} {atom.coord[0]:.6f} {atom.coord[1]:.6f} {atom.coord[2]:.6f}"
            lines.append(line)
        return "\n".join(lines)
    

## Basic usage of `Molecule` class


In [ ]:
benzene_xyz = """6
Benzene molecule
C 0.000000 1.402720 0.000000
C 1.214790 0.701360 0.000000
C 1.214790 -0.701360 0.000000
C 0.000000 -1.402720 0.000000
C -1.214790 -0.701360 0.000000
C -1.214790 0.701360 0.000000
H 0.000000 2.490290 0.000000
H 2.156660 1.245150 0.000000
H 2.156660 -1.245150 0.000000
H 0.000000 -2.490290 0.000000
H -2.156660 -1.245150 0.000000
H -2.156660 1.245150 0.000000
"""
benzene = Molecule.from_string(benzene_xyz)
print(benzene.to_string())
print(f"Total electrons in benzene: {benzene.n_electrons}")

### Visualizing an Atom-Based Molecule with `py3Dmol`

The following example creates a small water molecule from `Atom` instances and displays it interactively.

In [ ]:
import py3Dmol
def view_molecule(molecule: Molecule):
    xyz = molecule.to_string()
    view = py3Dmol.view(width=400, height=400)
    view.addModel(xyz, "xyz")
    view.setStyle({'stick': {}})
    view.zoomTo()
    return view 

In [ ]:
view = view_molecule(benzene)
view.show()

## Creating a Python Package

After defining the `Atom` and `Molecule` classes, we can organize the code into a Python package.

In this project, we use the `src` layout:

```text
src/
    theochem2026/
        __init__.py
        atom.py
        molecule.py
```
Here, theochem2026 is the package name, and atom.py and molecule.py are modules inside that package.
The file __init__.py marks the directory as a package and can also re-export classes for convenient imports.

After installing the project in editable mode (pip install -e .), we can import classes in any notebook:

```text
from theochem2026 import Atom, Molecule
```

or directly from modules:

```text
from theochem2026.atom import Atom
from theochem2026.molecule import Molecule
```

This allows us to create Atom and Molecule objects and use their methods in notebooks and scripts.

Organizing our code into a package helps to make it reusable and it allows us to easily share our code with others. __init__.py can also be used to define the public API of the package by specifying which classes and functions should be accessible when the package is imported. For example, we could add the following lines to __init__.py:

from .atom import Atom
from .molecule import Molecule


This way, when we import the package using from theochem2026 import *, both Atom and Molecule will be available in the namespace. 


pyproject.toml can be used to define the package metadata and dependencies, making it easier to manage and distribute the package. Installation can be done using pip, and the package can be published to PyPI for others to use. In order to make the package installable, we can create a setup.py file that uses setuptools to define the package metadata and dependencies. For example:
from setuptools import setup, find_packages     